# Attention Mechanisms from Scratch

**Goal:** Implement scaled dot-product attention, multi-head attention, self-attention, cross-attention, and causal masking — all in NumPy.

## Core Formula: Scaled Dot-Product Attention

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right) V$$

where $Q \in \mathbb{R}^{n \times d_k}$, $K \in \mathbb{R}^{m \times d_k}$, $V \in \mathbb{R}^{m \times d_v}$.

- $n$ = number of query positions, $m$ = number of key/value positions
- $d_k$ = key dimension, $d_v$ = value dimension

### Why scale by $\sqrt{d_k}$?

If $q$ and $k$ are random vectors with entries $\sim \mathcal{N}(0, 1)$, then $q \cdot k = \sum_{i=1}^{d_k} q_i k_i$ has variance $d_k$. For large $d_k$, the dot products become large, pushing softmax into regions with tiny gradients. Dividing by $\sqrt{d_k}$ normalizes the variance to 1.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
np.random.seed(42)
np.set_printoptions(precision=4, suppress=True)

## 1. Softmax (stable)

In [ ]:
def softmax(x, axis=-1):
    """Numerically stable softmax."""
    e = np.exp(x - x.max(axis=axis, keepdims=True))
    return e / e.sum(axis=axis, keepdims=True)

## 2. Scaled Dot-Product Attention

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}} + M\right) V$$

where $M$ is an optional mask (e.g., $-\infty$ for causal masking or padding).

In [ ]:
def scaled_dot_product_attention(Q, K, V, mask=None):
    """Scaled dot-product attention.
    
    Args:
        Q: (..., n, d_k) queries
        K: (..., m, d_k) keys
        V: (..., m, d_v) values
        mask: (..., n, m) additive mask. Use -inf to block positions.
    Returns:
        output: (..., n, d_v)
        weights: (..., n, m) attention weights
    """
    d_k = Q.shape[-1]
    
    # Compute attention scores
    scores = Q @ K.swapaxes(-2, -1) / np.sqrt(d_k)  # (..., n, m)
    
    # Apply mask (if any)
    if mask is not None:
        scores = scores + mask
    
    # Softmax over keys dimension
    weights = softmax(scores, axis=-1)  # (..., n, m)
    
    # Weighted sum of values
    output = weights @ V  # (..., n, d_v)
    
    return output, weights

# Demo
n, m, d_k, d_v = 4, 6, 8, 8
Q = np.random.randn(n, d_k)
K = np.random.randn(m, d_k)
V = np.random.randn(m, d_v)

output, weights = scaled_dot_product_attention(Q, K, V)
print(f"Q: {Q.shape}, K: {K.shape}, V: {V.shape}")
print(f"Output: {output.shape}, Weights: {weights.shape}")
print(f"Weights sum per query (should be 1): {weights.sum(axis=-1)}")

## 3. Visualize Attention Weights

In [ ]:
def plot_attention(weights, x_labels=None, y_labels=None, title='Attention Weights', ax=None):
    """Visualize attention weight matrix."""
    if ax is None:
        fig, ax = plt.subplots(figsize=(6, 5))
    im = ax.imshow(weights, cmap='Blues', vmin=0)
    ax.set_xlabel('Keys')
    ax.set_ylabel('Queries')
    ax.set_title(title)
    if x_labels is not None:
        ax.set_xticks(range(len(x_labels)))
        ax.set_xticklabels(x_labels, rotation=45, ha='right')
    if y_labels is not None:
        ax.set_yticks(range(len(y_labels)))
        ax.set_yticklabels(y_labels)
    plt.colorbar(im, ax=ax, fraction=0.046)
    return ax

# Attention with semantic meaning
query_words = ['the', 'cat', 'sat', 'down']
key_words = ['a', 'small', 'cat', 'sat', 'on', 'the']

# Create embeddings (random but make 'cat' and 'sat' similar in Q and K)
d = 16
Q_words = np.random.randn(4, d) * 0.5
K_words = np.random.randn(6, d) * 0.5
V_words = np.random.randn(6, d)

# Make Q[1] (cat) similar to K[2] (cat), and Q[2] (sat) similar to K[3] (sat)
K_words[2] = Q_words[1] + np.random.randn(d) * 0.1
K_words[3] = Q_words[2] + np.random.randn(d) * 0.1
# Make Q[0] (the) similar to K[5] (the)
K_words[5] = Q_words[0] + np.random.randn(d) * 0.1

_, attn_weights = scaled_dot_product_attention(Q_words, K_words, V_words)

fig, ax = plt.subplots(figsize=(7, 5))
plot_attention(attn_weights, x_labels=key_words, y_labels=query_words, 
              title='Attention: queries attend to matching keys', ax=ax)
plt.tight_layout()
plt.show()

## 4. Demonstrate the Scaling Effect

Show what happens with and without $\sqrt{d_k}$ scaling.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for i, d_k_test in enumerate([8, 64, 512]):
    Q_test = np.random.randn(10, d_k_test)
    K_test = np.random.randn(10, d_k_test)
    
    # Without scaling
    scores_raw = Q_test @ K_test.T
    weights_raw = softmax(scores_raw)
    
    # With scaling
    scores_scaled = scores_raw / np.sqrt(d_k_test)
    weights_scaled = softmax(scores_scaled)
    
    # Entropy as measure of peakiness (higher = more uniform)
    entropy_raw = -np.sum(weights_raw * np.log(weights_raw + 1e-12), axis=-1).mean()
    entropy_scaled = -np.sum(weights_scaled * np.log(weights_scaled + 1e-12), axis=-1).mean()
    
    axes[i].bar(['Unscaled', 'Scaled'], [entropy_raw, entropy_scaled], 
                color=['salmon', 'steelblue'])
    axes[i].set_title(f'd_k = {d_k_test}')
    axes[i].set_ylabel('Avg Entropy')
    axes[i].set_ylim(0, np.log(10) + 0.2)
    axes[i].axhline(y=np.log(10), color='gray', linestyle='--', label='Max entropy (uniform)')
    axes[i].legend(fontsize=8)

plt.suptitle('Effect of sqrt(d_k) scaling on attention entropy', fontsize=13)
plt.tight_layout()
plt.show()
print("Without scaling, large d_k causes near-zero entropy (one-hot attention).")
print("Scaling preserves a reasonable entropy, allowing meaningful gradients.")

## 5. Self-Attention

In self-attention, Q, K, and V all come from the **same** input sequence, projected through learned weight matrices:

$$Q = XW_Q, \quad K = XW_K, \quad V = XW_V$$

where $X \in \mathbb{R}^{n \times d_{model}}$.

In [ ]:
class SelfAttention:
    """Single-head self-attention layer."""
    
    def __init__(self, d_model, d_k, d_v):
        scale = np.sqrt(2.0 / d_model)
        self.W_Q = np.random.randn(d_model, d_k) * scale
        self.W_K = np.random.randn(d_model, d_k) * scale
        self.W_V = np.random.randn(d_model, d_v) * scale
        self.d_k = d_k
    
    def forward(self, X, mask=None):
        """Self-attention forward.
        Args:
            X: (batch, seq_len, d_model)
            mask: optional (batch, seq_len, seq_len)
        Returns:
            output: (batch, seq_len, d_v)
            weights: (batch, seq_len, seq_len)
        """
        self.X = X
        self.Q = X @ self.W_Q  # (batch, n, d_k)
        self.K = X @ self.W_K  # (batch, n, d_k)
        self.V = X @ self.W_V  # (batch, n, d_v)
        
        output, weights = scaled_dot_product_attention(self.Q, self.K, self.V, mask)
        self.weights = weights
        return output, weights
    
    def backward(self, dout):
        """Self-attention backward.
        Args:
            dout: (batch, n, d_v)
        Returns:
            dX: (batch, n, d_model)
            dW_Q, dW_K, dW_V
        """
        # dout -> d(weights @ V)
        # d_weights = dout @ V^T, dV = weights^T @ dout
        dV = self.weights.swapaxes(-2, -1) @ dout  # (batch, n, d_v)
        d_weights = dout @ self.V.swapaxes(-2, -1)  # (batch, n, n)
        
        # Softmax backward: d_scores = weights * (d_weights - sum(d_weights * weights, axis=-1, keepdims=True))
        sum_dw = (d_weights * self.weights).sum(axis=-1, keepdims=True)
        d_scores = self.weights * (d_weights - sum_dw) / np.sqrt(self.d_k)
        
        # d_scores = dQ @ K^T / sqrt(d_k) -> dQ, dK
        dQ = d_scores @ self.K   # (batch, n, d_k)
        dK = d_scores.swapaxes(-2, -1) @ self.Q  # (batch, n, d_k)
        
        # Through projections
        # Q = X @ W_Q -> dX_Q = dQ @ W_Q^T, dW_Q = X^T @ dQ
        dX = dQ @ self.W_Q.T + dK @ self.W_K.T + dV @ self.W_V.T
        
        # Sum over batch for parameter gradients
        dW_Q = np.sum(self.X.swapaxes(-2, -1) @ dQ, axis=0)
        dW_K = np.sum(self.X.swapaxes(-2, -1) @ dK, axis=0)
        dW_V = np.sum(self.X.swapaxes(-2, -1) @ dV, axis=0)
        
        return dX, dW_Q, dW_K, dW_V

# Test self-attention
batch, seq_len, d_model = 2, 5, 16
d_k = d_v = 8
X_test = np.random.randn(batch, seq_len, d_model)

sa = SelfAttention(d_model, d_k, d_v)
out, weights = sa.forward(X_test)
print(f"Input: {X_test.shape}")
print(f"Output: {out.shape}")
print(f"Weights: {weights.shape}")
print(f"Attention weights (batch 0):\n{weights[0]}")

## 6. Cross-Attention

In cross-attention, queries come from one sequence and keys/values from another (e.g., decoder attends to encoder in machine translation).

$$Q = X_{dec} W_Q, \quad K = X_{enc} W_K, \quad V = X_{enc} W_V$$

In [ ]:
class CrossAttention:
    """Cross-attention: queries from one source, keys/values from another."""
    
    def __init__(self, d_model, d_k, d_v):
        scale = np.sqrt(2.0 / d_model)
        self.W_Q = np.random.randn(d_model, d_k) * scale
        self.W_K = np.random.randn(d_model, d_k) * scale
        self.W_V = np.random.randn(d_model, d_v) * scale
    
    def forward(self, X_query, X_kv, mask=None):
        """Cross-attention forward.
        Args:
            X_query: (batch, n, d_model) — decoder states
            X_kv:    (batch, m, d_model) — encoder states
        Returns:
            output: (batch, n, d_v)
            weights: (batch, n, m)
        """
        Q = X_query @ self.W_Q  # (batch, n, d_k)
        K = X_kv @ self.W_K     # (batch, m, d_k)
        V = X_kv @ self.W_V     # (batch, m, d_v)
        return scaled_dot_product_attention(Q, K, V, mask)

# Demo: encoder has 8 tokens, decoder has 4 tokens
X_enc = np.random.randn(1, 8, 16)
X_dec = np.random.randn(1, 4, 16)

ca = CrossAttention(d_model=16, d_k=8, d_v=8)
out_cross, weights_cross = ca.forward(X_dec, X_enc)
print(f"Decoder queries: {X_dec.shape}")
print(f"Encoder keys/values: {X_enc.shape}")
print(f"Cross-attention output: {out_cross.shape}")
print(f"Cross-attention weights: {weights_cross.shape} (4 decoder queries x 8 encoder keys)")

## 7. Causal (Masked) Attention

In autoregressive models (GPT-style), position $t$ can only attend to positions $\leq t$. We achieve this by adding $-\infty$ to the upper triangle of the score matrix before softmax.

In [ ]:
def causal_mask(seq_len):
    """Create causal attention mask.
    Returns (seq_len, seq_len) with 0 on valid positions and -inf on future positions.
    """
    mask = np.triu(np.ones((seq_len, seq_len)) * (-1e9), k=1)
    return mask

# Demo
seq_len = 6
mask = causal_mask(seq_len)
print("Causal mask (0 = attend, -inf = block):")
print(np.where(mask < -1, '-inf', '  0 '))

# Self-attention with causal mask
X_causal = np.random.randn(1, seq_len, 16)
sa_causal = SelfAttention(d_model=16, d_k=8, d_v=8)

out_causal, weights_causal = sa_causal.forward(X_causal, mask=mask)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Normal self-attention
out_normal, weights_normal = sa_causal.forward(X_causal, mask=None)
plot_attention(weights_normal[0], title='Self-Attention (no mask)', ax=axes[0])

# Causal self-attention
plot_attention(weights_causal[0], title='Causal Self-Attention (masked)', ax=axes[1])

plt.tight_layout()
plt.show()
print("Notice: in causal attention, each position only attends to itself and earlier positions.")
print("Upper triangle is zero (future tokens blocked).")

## 8. Multi-Head Attention

Instead of a single attention function, we project Q, K, V into $h$ different subspaces (heads), compute attention in each, concatenate, and project back:

$$\text{MultiHead}(Q, K, V) = \text{Concat}(\text{head}_1, \ldots, \text{head}_h) W_O$$
$$\text{head}_i = \text{Attention}(QW_Q^i, KW_K^i, VW_V^i)$$

Each head uses $d_k = d_v = d_{model} / h$, so total computation is similar to single-head with full dimensionality.

In [ ]:
class MultiHeadAttention:
    """Multi-head attention."""
    
    def __init__(self, d_model, n_heads):
        assert d_model % n_heads == 0
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_k = d_model // n_heads
        
        scale = np.sqrt(2.0 / d_model)
        # Combined projections for all heads
        self.W_Q = np.random.randn(d_model, d_model) * scale
        self.W_K = np.random.randn(d_model, d_model) * scale
        self.W_V = np.random.randn(d_model, d_model) * scale
        self.W_O = np.random.randn(d_model, d_model) * scale
    
    def _split_heads(self, X):
        """(batch, seq, d_model) -> (batch, n_heads, seq, d_k)"""
        batch, seq_len, _ = X.shape
        X = X.reshape(batch, seq_len, self.n_heads, self.d_k)
        return X.transpose(0, 2, 1, 3)
    
    def _merge_heads(self, X):
        """(batch, n_heads, seq, d_k) -> (batch, seq, d_model)"""
        batch, _, seq_len, _ = X.shape
        X = X.transpose(0, 2, 1, 3)  # (batch, seq, n_heads, d_k)
        return X.reshape(batch, seq_len, self.d_model)
    
    def forward(self, X_q, X_kv=None, mask=None):
        """Multi-head attention forward.
        
        Args:
            X_q: (batch, n, d_model) — queries
            X_kv: (batch, m, d_model) — keys/values. If None, self-attention.
            mask: optional (n, m) or (batch, n, m)
        Returns:
            output: (batch, n, d_model)
            weights: (batch, n_heads, n, m)
        """
        if X_kv is None:
            X_kv = X_q  # self-attention
        
        # Project
        Q = X_q @ self.W_Q    # (batch, n, d_model)
        K = X_kv @ self.W_K   # (batch, m, d_model)
        V = X_kv @ self.W_V   # (batch, m, d_model)
        
        # Split into heads
        Q = self._split_heads(Q)  # (batch, n_heads, n, d_k)
        K = self._split_heads(K)  # (batch, n_heads, m, d_k)
        V = self._split_heads(V)  # (batch, n_heads, m, d_k)
        
        # Attention per head
        attn_out, weights = scaled_dot_product_attention(Q, K, V, mask)
        # attn_out: (batch, n_heads, n, d_k)
        
        # Merge heads and project
        concat = self._merge_heads(attn_out)  # (batch, n, d_model)
        output = concat @ self.W_O             # (batch, n, d_model)
        
        self.weights = weights
        return output, weights

# Test MHA
d_model = 32
n_heads = 4
batch, seq_len = 2, 6

mha = MultiHeadAttention(d_model, n_heads)
X_mha = np.random.randn(batch, seq_len, d_model)

out_mha, weights_mha = mha.forward(X_mha, mask=causal_mask(seq_len))
print(f"Input: {X_mha.shape}")
print(f"Output: {out_mha.shape}")
print(f"Weights: {weights_mha.shape} (batch, heads, queries, keys)")

## 9. Visualize Multi-Head Attention Patterns

In [ ]:
# Visualize each head's attention pattern
tokens = ['The', 'cat', 'sat', 'on', 'the', 'mat']
X_vis = np.random.randn(1, 6, d_model)
_, weights_vis = mha.forward(X_vis, mask=causal_mask(6))

fig, axes = plt.subplots(1, n_heads, figsize=(4 * n_heads, 4))
for h in range(n_heads):
    im = axes[h].imshow(weights_vis[0, h], cmap='Blues', vmin=0, vmax=1)
    axes[h].set_xticks(range(6))
    axes[h].set_xticklabels(tokens, rotation=45, ha='right', fontsize=9)
    axes[h].set_yticks(range(6))
    axes[h].set_yticklabels(tokens, fontsize=9)
    axes[h].set_title(f'Head {h}')

plt.suptitle('Multi-Head Causal Self-Attention (each head learns different patterns)', fontsize=13)
plt.tight_layout()
plt.show()

## 10. Comparing Attention Variants Side by Side

In [ ]:
# All three attention types in one view
d_model_demo = 16
n_seq = 5
m_seq = 7

X_self = np.random.randn(1, n_seq, d_model_demo)
X_enc_demo = np.random.randn(1, m_seq, d_model_demo)
X_dec_demo = np.random.randn(1, n_seq, d_model_demo)

# Self-attention (no mask)
sa_demo = SelfAttention(d_model_demo, 8, 8)
_, w_self = sa_demo.forward(X_self)

# Self-attention (causal)
_, w_causal_demo = sa_demo.forward(X_self, mask=causal_mask(n_seq))

# Cross-attention
ca_demo = CrossAttention(d_model_demo, 8, 8)
_, w_cross_demo = ca_demo.forward(X_dec_demo, X_enc_demo)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

plot_attention(w_self[0], title='Self-Attention\n(encoder-style, bidirectional)', ax=axes[0])
plot_attention(w_causal_demo[0], title='Causal Self-Attention\n(decoder-style, unidirectional)', ax=axes[1])
plot_attention(w_cross_demo[0], title=f'Cross-Attention\n({n_seq} decoder queries x {m_seq} encoder keys)', ax=axes[2])

plt.tight_layout()
plt.show()

## 11. Attention Backward Pass — Gradient Check

In [ ]:
# Gradient check for SelfAttention
sa_check = SelfAttention(d_model=8, d_k=4, d_v=4)
X_check = np.random.randn(1, 3, 8)

out_check, _ = sa_check.forward(X_check)
dout_check = np.random.randn(*out_check.shape)

dX_anal, dW_Q_anal, dW_K_anal, dW_V_anal = sa_check.backward(dout_check)

# Numerical gradient for dX
eps = 1e-5
dX_num = np.zeros_like(X_check)
for i in range(X_check.shape[1]):
    for j in range(X_check.shape[2]):
        X_plus = X_check.copy(); X_plus[0, i, j] += eps
        X_minus = X_check.copy(); X_minus[0, i, j] -= eps
        out_plus, _ = sa_check.forward(X_plus)
        out_minus, _ = sa_check.forward(X_minus)
        dX_num[0, i, j] = np.sum((out_plus - out_minus) * dout_check) / (2 * eps)

print("Self-Attention Gradient Check (dX):")
print(f"  Analytical (first row): {dX_anal[0, 0, :4]}")
print(f"  Numerical  (first row): {dX_num[0, 0, :4]}")
rel_err = np.abs(dX_anal - dX_num) / (np.abs(dX_num) + 1e-8)
print(f"  Max relative error: {rel_err.max():.2e}")

## Interview Questions

### Q: "Why scale by $\sqrt{d_k}$?"

If $q, k \in \mathbb{R}^{d_k}$ with entries $\sim \mathcal{N}(0, 1)$, then $q \cdot k$ has mean 0 and variance $d_k$. For large $d_k$ (e.g., 512), the raw dot products are very large, causing softmax to produce near-one-hot distributions. This means:
1. Gradients through softmax are near-zero (saturation)
2. The model cannot spread attention across multiple keys

Dividing by $\sqrt{d_k}$ normalizes the variance to 1, keeping softmax in a well-behaved regime.

### Q: "Multi-head vs single-head attention?"

Multi-head attention lets the model jointly attend to information from **different representation subspaces** at different positions. A single head can only compute one attention pattern.

With $h$ heads of dimension $d_k = d_{model}/h$:
- Each head can learn a different "type" of relationship (syntactic, semantic, positional, etc.)
- Total compute is the same as single-head with $d_k = d_{model}$ (just reorganized)
- The output projection $W_O$ learns to combine head outputs

### Q: "Attention complexity?"

For sequence length $n$ and dimension $d$:
- **Time:** $O(n^2 d)$ — computing $QK^T$ is $(n \times d) \cdot (d \times n) = O(n^2 d)$
- **Memory:** $O(n^2)$ — storing the attention weight matrix

This quadratic scaling in $n$ is the main bottleneck for long sequences. Efficient variants:
- **Sparse attention** (BigBird, Longformer): $O(n \sqrt{n})$ or $O(n \log n)$
- **Linear attention** (Performer): $O(nd)$ by approximating softmax with kernels
- **Flash Attention**: same $O(n^2 d)$ complexity but optimized memory access patterns (IO-aware)

### Q: "Self-attention vs cross-attention?"

| | Self-Attention | Cross-Attention |
|---|---------------|----------------|
| Q, K, V source | Same sequence | Q from one, K/V from another |
| Use case | Encoder, decoder masked | Decoder attending to encoder |
| Weight matrix | $QK^T$ is $(n \times n)$ | $QK^T$ is $(n_{dec} \times n_{enc})$ |

### Q: "What is the role of attention in transformers vs RNNs?"

- RNNs compress the entire past into a fixed-size hidden state (information bottleneck)
- Attention allows **direct access** to all past representations, with learned weighting
- Self-attention also enables **parallel** processing of all positions (no sequential dependency during forward pass)
- This is why Transformers train much faster than RNNs on modern hardware